In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch
from torch import nn, optim
import argparse
import sys
sys.path.append('./')
sys.path.append('../')
sys.path.append('../../')
sys.path.append('../../../')
import os
import yaml
from easydict import EasyDict
from collections import OrderedDict
import random
import numpy as np
import pickle as pkl
import h5py

from torch.utils.data import IterableDataset

from Data.Transition1x import generate_dataloader_dynamics
from Model.model import DistFlowMatchingNetwork, ODEWrapper, InterpNetwork
from Model.backbone import generate_backbone
from Model.head import generate_head
from Model.model import MDNet
from Utils import get_logger, get_new_log_dir, seed_all, Kabsch_alignment, rmsd_loss, generate_fully_connected, create_angular_index, calculate_angle, d_mae_loss, calculate_efh, AU2EV, MLCalculator, PyscfCalculator, neb, Sella_Opt, count_negative_eig, construct_atoms
import gc

from copy import deepcopy

from torch_scatter import scatter_mean, scatter_add
from torch.optim import LBFGS

from torch_geometric.data import Data, DataLoader

import pickle

from sella import Sella

In [ ]:
from ase import Atoms
from ase.mep.neb import NEB, NEBTools, NEBOptimizer
from ase.optimize import MDMin, BFGS
from ase.calculators.calculator import Calculator, all_changes

In [ ]:
parser = argparse.ArgumentParser(description='Training Transition1x dynamics')
parser.add_argument('--config_file_potential', required=True)
parser.add_argument('--log_prefix', default='logs')
parser.add_argument('--notes', default=' ')
parser.add_argument('--device', default='cuda')
parser.add_argument('--resume_status', default=' ')
parser.add_argument('--potential', default=' ')
args = parser.parse_args(['--device', 'cuda',
                          '--config_file_potential', '../../../Configs/Potential.yml',
                          '--potential', '']) # path to trained MLIP checkpoint

In [ ]:
def calculate_loss_coord(adj_matrix, coord, edge_index):
    src = edge_index[0]
    dst = edge_index[1]
    diff = coord[src] - coord[dst]
    dists = torch.norm(diff, p=2, dim=-1)
    loss = torch.sum(torch.square(dists - adj_matrix) * torch.pow(1 / (adj_matrix + 1e-6), 2))
    return loss

def calculate_loss_angle(angle_vals, coord, angle_index):
    ang_i, ang_j, ang_k = angle_index
    ang_curr = calculate_angle(coord, ang_i, ang_j, ang_k)
    dist_ij = torch.norm(coord[ang_i] - coord[ang_j], p=2, dim=-1)
    dist_jk = torch.norm(coord[ang_j] - coord[ang_k], p=2, dim=-1)
    loss = torch.sum(torch.square(angle_vals - ang_curr) * torch.pow(1 / (dist_ij + 1e-6), 1) * torch.pow(1 / (dist_jk + 1e-6), 1))
    return loss

In [ ]:
dtype = torch.float32

config_path=args.config_file_potential
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
config = EasyDict(config)
config.notes = args.notes

device = args.device

In [ ]:
seed_all(config.train.seed)
torch.backends.cudnn.benchmark = True

In [ ]:
backbone = generate_backbone(config.model.backbone)
head = generate_head(config.model.head)

In [ ]:
REFERENCE_ENERGIES = {
    1: -13.62222753701504,
    6: -1029.4130839658328,
    7: -1484.8710358098756,
    8: -2041.8396277138045,
    9: -2712.8213146878606,
}

In [ ]:
potential_model = MDNet(backbone, head, REFERENCE_ENERGIES)
best_state = torch.load(args.potential, map_location=device)
potential_model.load_state_dict(best_state['model'])
potential_model.eval()

In [ ]:
potential_model = potential_model.to(device)

In [ ]:
potential_model.eval()

In [ ]:
calculator_ml = MLCalculator(potential_model)
calculator_dft = PyscfCalculator(device='cuda')

In [ ]:
with open('res_reactdiff.pickle', 'rb') as f:
    temp = pickle.load(f)

In [ ]:
temp.keys()

In [ ]:
energy_diffs_dft = []
rmsds = []
dmaes = []
res_dft = []
res_dft_orig = []
success = []
for i in range(len(temp['rmsd'])):
    energy_diff = temp['true_energy_barrier_reactant'][i]
    transition_state_pos = temp['pred_transition_state_pos'][i]
    atom_type = temp['atom_types'][i]
    reactant_pos = temp['reactant_product_pos'][i][0]
    product_pos = temp['reactant_product_pos'][i][1]

    atom = construct_atoms(atom_type, transition_state_pos)
    dft_orig = calculate_efh(atom, f=True, hess=True, return_metrics=True)
    res_dft_orig.append(dft_orig)
    energy_trans_orig = dft_orig[0].e_tot * AU2EV

    atom_configs, is_success_neb = neb(calculator_ml, atom_type, reactant_pos, product_pos, transition_state_pos, log = '-')
    success.append(is_success_neb)

    max_energy_ind = 0
    max_energy = -10000000.0

    for j in range(len(atom_configs)):
        atoms = atom_configs[j]
        x = torch.tensor(atoms.get_atomic_numbers()).to(device)
        pos = torch.tensor(atoms.get_positions(), dtype=torch.float32).to(device)
        batch = torch.zeros_like(x).to(device)
        energy, force = potential_model.get_energy_and_force(x, pos, None, None, batch)
        energy = energy.cpu()
        if energy > max_energy:
            max_energy = energy
            max_energy_ind = j

    optimized_atom = atom_configs[max_energy_ind]

    loss_pos_rmse = rmsd_loss(torch.tensor(optimized_atom.get_positions()), temp['true_transition_state_pos'][i].cpu(), torch.zeros_like(atom_type, dtype=torch.long, device='cpu'))
    loss_pos_dmae_temp = d_mae_loss(torch.tensor(optimized_atom.get_positions()), temp['true_transition_state_pos'][i].cpu(), torch.zeros_like(atom_type, dtype=torch.long, device='cpu'))

    dft = calculate_efh(optimized_atom, f=True, hess=True, return_metrics=True)
    res_dft.append(dft)
    reactant_atom = construct_atoms(atom_type, temp['reactant_product_pos'][i][0])
    energy_reactant = calculate_efh(reactant_atom, f=False, hess=False)[0].e_tot * AU2EV
    energy_trans = dft[0].e_tot * AU2EV
    loss_energy_temp_dft = abs((energy_reactant - energy_trans) - energy_diff)

    energy_diffs_dft.append(loss_energy_temp_dft)
    rmsds.append(loss_pos_rmse)
    dmaes.append(loss_pos_dmae_temp)

In [ ]:
import pickle
with open('optimized_res_oa_reactdiff.pickle', 'wb') as f:
    pickle.dump({
        'hess':{
            'rmsd': rmsds,
            'dmae': dmaes,
            'energy_diff': energy_diffs_dft,
            'success': success,
            'dft_res': res_dft,
            'dft_res_orig': res_dft_orig
        }
    }, f)